# v8: Hidden histograms and shape features

**Version:** v8  
**Kaggle submission:** none yet (files `v8_*` are superseded by v9, which only improves the cost model's capacity)

**Hypothesis:** the 170 sensors have structure. Raw bin counts of a histogram mix 'how much the truck was used' with 'how its usage was distributed'; trees struggle to separate the two. If there are more histograms than the one we found in v1, **shape features** (bin shares, mean, entropy, tails) should add real signal, for the failure model and/or the repair-cost model.

*Self-contained: needs only `../data/train.csv` and `../data/test.csv`. Part of the Fleet Maintenance project (Kaggle `mlprojectfleet2`); the cost, decision rule and validation protocol are derived in `fleet_maintenance.ipynb`.*

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, xgboost as xgb
from pathlib import Path
from scipy.stats import ks_2samp
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, r2_score, brier_score_loss, log_loss
from sklearn.model_selection import StratifiedKFold, KFold, train_test_split, cross_val_predict

DATA, OUT = Path("../data"), Path("outputs"); OUT.mkdir(exist_ok=True)
SEED = 42
INSPECT, UNDER, OVER = 10.0, 0.20, 0.02          # inspection cost; shortage cost / unit; unused-budget cost / unit
Q_STAR = UNDER / (UNDER + OVER)                  # newsvendor critical ratio 0.909
HISTB = [f"s{i:03d}" for i in range(54, 64)]     # 10-bin histogram that is shifted by one bin in test

def load(raw=False):
    """train/test with the v1 fixes: realign s054-s063 (test bin k+1 == train bin k), drop s047 (all NaN in
    test) and s146 (constant). raw=True returns the files untouched."""
    tr, te = pd.read_csv(DATA / "train.csv"), pd.read_csv(DATA / "test.csv")
    if not raw:
        te[HISTB[:-1]] = te[HISTB[1:]].to_numpy(); te[HISTB[-1]] = np.nan
        tr, te = tr.drop(columns=["s047", "s146"]), te.drop(columns=["s047", "s146"])
    return tr, te, [c for c in tr.columns if c.startswith("s")]

tr, te, S = load()
dev_idx, ho_idx = train_test_split(np.arange(len(tr)), test_size=0.2, stratify=tr.failure, random_state=SEED)
dev, ho = tr.iloc[dev_idx].reset_index(drop=True), tr.iloc[ho_idx].reset_index(drop=True)   # ho: locked 20% hold-out
y, c = dev.failure.to_numpy(), dev.repair_cost.fillna(0).to_numpy() * dev.failure.to_numpy()
y_ho, c_ho = ho.failure.to_numpy(), ho.repair_cost.fillna(0).to_numpy() * ho.failure.to_numpy()
c_all = tr.repair_cost.fillna(0).to_numpy() * tr.failure.to_numpy()
F = list(StratifiedKFold(5, shuffle=True, random_state=SEED).split(np.zeros(len(y)), y))   # the 5 fixed folds
print(f"train {tr.shape}  test {te.shape}  sensors {len(S)}  dev {len(dev)} / hold-out {len(ho)}  failures {int(y.sum())}")

train (56000, 172)  test (20000, 169)  sensors 168  dev 44800 / hold-out 11200  failures 318


In [2]:
def vehicle_cost(c, flag, b):
    return np.where(flag == 1, INSPECT + UNDER * np.maximum(c - b, 0) + OVER * np.maximum(b - c, 0), c)

def per_vehicle(c, flag, b):
    return float(vehicle_cost(np.asarray(c, float), np.asarray(flag), np.asarray(b, float)).mean())

fold_cost = lambda flag, b: np.array([per_vehicle(c[v], flag[v], b[v]) for _, v in F])

def write_submission(ids, flag, budget, name):
    flag, budget = np.asarray(flag).astype(int), np.maximum(np.asarray(budget, float), 0)
    assert len(ids) == 20000 and ids.is_unique and set(flag) <= {0, 1} and np.isfinite(budget).all()
    pd.DataFrame({"id": ids, "flag": flag, "budget": budget.round(2)}).to_csv(OUT / f"submission_{name}.csv", index=False)
    print(f"wrote outputs/submission_{name}.csv: flagged {flag.mean():.2%} of test vehicles")

def decide_global(p, fail_costs):
    """Same repair-cost distribution F (empirical failure costs) for every vehicle."""
    p = np.clip(np.asarray(p, float), 0, 1); fc = np.sort(np.asarray(fail_costs, float)); mu = fc.mean()
    b = np.zeros_like(p); need = p > 1 - Q_STAR
    b[need] = np.quantile(fc, 1 - (1 - Q_STAR) / p[need])                       # newsvendor budget
    fail = (UNDER * np.maximum(fc[None, :] - b[:, None], 0) + OVER * np.maximum(b[:, None] - fc[None, :], 0)).mean(1)
    return ((INSPECT + p * fail + (1 - p) * OVER * b) < p * mu).astype(int), b   # flag iff cheaper than not flagging

def decide_pv(p, m, resid, fail_costs=None, K=99, scale=None):
    """Per-vehicle cost distribution: log c | fail ~ m_i + scale_i * residuals (out-of-fold residuals).
    fail_costs given -> budget from that GLOBAL distribution (ablation); scale -> per-vehicle spread (v9)."""
    p, m = np.clip(np.asarray(p, float), 0, 1), np.asarray(m, float)
    sc = np.ones_like(m) if scale is None else np.asarray(scale, float)
    M = np.exp(m[:, None] + sc[:, None] * np.quantile(resid, (np.arange(K) + 0.5) / K)[None, :]); mu = M.mean(1)
    b = np.zeros_like(p); need = p > 1 - Q_STAR; level = 1 - (1 - Q_STAR) / p[need]
    b[need] = (np.exp(m[need] + sc[need] * np.quantile(resid, level)) if fail_costs is None
               else np.quantile(np.asarray(fail_costs, float), level))
    fail = (UNDER * np.maximum(M - b[:, None], 0) + OVER * np.maximum(b[:, None] - M, 0)).mean(1)
    return ((INSPECT + p * fail + (1 - p) * OVER * b) < p * mu).astype(int), b

class Platt:
    def fit(self, p, y):
        self.m = LogisticRegression(C=1e6, max_iter=1000).fit(self._x(p), y); return self
    def predict(self, p): return self.m.predict_proba(self._x(p))[:, 1]
    @staticmethod
    def _x(p): p = np.clip(p, 1e-6, 1 - 1e-6); return np.log(p / (1 - p))[:, None]

class Iso:
    def fit(self, p, y):
        self.m = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1).fit(p, y); return self
    def predict(self, p): return self.m.predict(p)

make_cal = lambda kind: {"platt": Platt, "iso": Iso}[kind]()

PARAMS = dict(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
              min_child_weight=1, n_jobs=-1, random_state=SEED)                         # failure classifier (v2..)
CM = dict(n_estimators=200, max_depth=3, learning_rate=0.03, subsample=0.8, colsample_bytree=0.5,
          min_child_weight=5, n_jobs=-1, random_state=SEED)                             # repair-cost model (v5)
inner = KFold(5, shuffle=True, random_state=SEED)

def cost_model(params, Xa, lc, Xb):
    """XGBoost on log repair cost (failures only). Returns predictions for Xb and OUT-OF-FOLD residuals of Xa."""
    resid = lc - cross_val_predict(xgb.XGBRegressor(**params), Xa, lc, cv=inner)
    return xgb.XGBRegressor(**params).fit(Xa, lc).predict(Xb), resid

def fit_oof(X, params=PARAMS):
    """5-fold out-of-fold failure probabilities on dev, and probabilities for test from a model fit on all train."""
    o = np.zeros(len(dev))
    for a, b in F:
        o[b] = xgb.XGBClassifier(**params).fit(X.iloc[a], y[a]).predict_proba(X.iloc[b])[:, 1]
    return o

shift = lambda p, k: k * p / (1 - p) / (1 + k * p / (1 - p))       # prior shift by odds factor k

In [3]:
HIST = {a: [f"s{i:03d}" for i in range(a, a + 10)] for a in (23, 37, 54, 80, 129, 154)}   # six hidden 10-bin histograms
HCOLS = {col for g in HIST.values() for col in g}

def shape(df):
    """Histogram-shape features: per-bin shares, mean/std bin, entropy, low/high-3 tail shares, log usage."""
    out = {"logU": np.log1p(df[HIST[54][:9]].sum(axis=1, min_count=9))}
    for a, g in HIST.items():
        g = g[:9] if a == 54 else g                        # test s063 is NaN after realignment
        x = df[g].to_numpy(float); tot = x.sum(1, keepdims=True)
        sh = np.where(tot > 0, x / np.where(tot > 0, tot, 1), np.nan); k = np.arange(len(g)); mean = (sh * k).sum(1)
        out.update({f"h{a}_sh{i}": sh[:, i] for i in range(len(g))})
        out[f"h{a}_mean"] = np.where(tot[:, 0] > 0, mean, np.nan)
        out[f"h{a}_std"] = np.where(tot[:, 0] > 0, np.sqrt(np.maximum((sh * k ** 2).sum(1) - mean ** 2, 0)), np.nan)
        with np.errstate(divide="ignore", invalid="ignore"):
            out[f"h{a}_ent"] = np.where(tot[:, 0] > 0, -np.nansum(np.where(sh > 0, sh * np.log(sh), 0), 1), np.nan)
        out[f"h{a}_lo3"], out[f"h{a}_hi3"] = sh[:, :3].sum(1), sh[:, -3:].sum(1)
    return pd.DataFrame(out, index=df.index)

def ratios(df, cols):
    U = df[HIST[54][:9]].sum(axis=1, min_count=9).replace(0, np.nan)
    return df[[col for col in cols if col not in HCOLS]].div(U, axis=0).add_suffix("_perU")

In [4]:
def pipeline_costs(oofs, base="A", cm=CM, X=None, repeats=None):
    """Paired CV cost for several failure-probability candidates through the v5 pipeline
    (nested cost model fitted once per fold, Platt fitted on the other folds' OOF)."""
    X = dev[S] if X is None else X
    cost = {k: [] for k in oofs}
    for a, b in F:
        fa = a[y[a] == 1]
        m_b, resid = cost_model(cm, X.iloc[fa], np.log(c[fa]), X.iloc[b])
        for k, o in oofs.items():
            fl, bu = decide_pv(make_cal("platt").fit(o[a], y[a]).predict(o[b]), m_b, resid)
            cost[k].append(per_vehicle(c[b], fl, bu))
    return {k: np.array(v) for k, v in cost.items()}

def report(cost, base):
    A = cost[base]
    for k, x in cost.items():
        d = x - A; print(f"{k:12s} CV cost {x.mean():.4f} ± {x.std():.3f}   paired vs {base}: {d.mean():+.4f}   per fold {np.round(d, 3)}   better {int((d < 0).sum())}/{len(d)}")

### 1. Discovering the hidden histograms
If a column set is a histogram over shared operating time, its row total is a fixed fraction of the known histogram's total. A non-negative least-squares fit of that total on all other columns exposes them.

In [5]:
from scipy.optimize import nnls
U = tr[HISTB].sum(axis=1, min_count=10); cand = [s for s in S if s not in HISTB]
ratio = tr[cand].div(U.replace(0, np.nan), axis=0)
cand = [s for s in cand if (ratio[s].dropna() <= 1.0001).mean() > 0.999 and tr[s].notna().mean() > 0.8]
d = pd.concat([tr[cand], U.rename("U")], axis=1).dropna().sample(15000, random_state=0)
coef, rn = nnls(d[cand].to_numpy(float), d.U.to_numpy(float))
print(f"relative residual of the fit: {rn / np.linalg.norm(d.U):.6f}  (exact)")
g = pd.Series(coef, index=cand).round(3); g = g[g > 0.05]
for v, grp in g.groupby(g): print(f"  equal coefficient {v}: {list(grp.index)}")
for a, cols in HIST.items():
    r = (tr[cols].sum(axis=1, min_count=10) / U).dropna()
    print(f"histogram s{a:03d}-s{a + 9:03d}: total / reference total = {r.median():.3f} (5-95%: {r.quantile(.05):.3f}-{r.quantile(.95):.3f})")

relative residual of the fit: 0.000006  (exact)
  equal coefficient 0.077: ['s080', 's081', 's082', 's083', 's084', 's085', 's086', 's087', 's088', 's089']
  equal coefficient 0.114: ['s167']
  equal coefficient 0.122: ['s129', 's130', 's131', 's132', 's133', 's134', 's135', 's136', 's137', 's138']
  equal coefficient 1.44: ['s023', 's024', 's025', 's026', 's027', 's028', 's029', 's030', 's031', 's032']
  equal coefficient 1.747: ['s154', 's155', 's156', 's158', 's159', 's160', 's161', 's162', 's163']
  equal coefficient 1.748: ['s157']
histogram s023-s032: total / reference total = 0.296 (5-95%: 0.296-0.297)
histogram s037-s046: total / reference total = 0.180 (5-95%: 0.180-0.181)
histogram s054-s063: total / reference total = 1.000 (5-95%: 1.000-1.000)
histogram s080-s089: total / reference total = 0.749 (5-95%: 0.747-0.752)
histogram s129-s138: total / reference total = 0.695 (5-95%: 0.693-0.698)
histogram s154-s163: total / reference total = 0.246 (5-95%: 0.246-0.247)


### 2. Are the new features safe under the depot shift?

In [6]:
def ks(x, z): x, z = x.dropna(), z.dropna(); return ks_2samp(x, z).statistic if len(x) and len(z) else np.nan
Str, Ste = shape(tr), shape(te)
print("same-bin KS of the five un-shifted histograms (max over bins):", {f"h{a}": round(float(max(ks(tr[col], te[col]) for col in cols)), 3) for a, cols in HIST.items() if a != 54})
sh = pd.Series({col: ks(Str[col], Ste[col]) for col in Str}); print(f"KS of the shape features: median {sh.median():.3f}  max {sh.max():.3f}")
usage = lambda df: df[HIST[54][:9]].sum(axis=1, min_count=9)
print(f"usage (operating-time total): train median {usage(tr).median():.3g}  test {usage(te).median():.3g}  KS {ks(usage(tr), usage(te)):.3f}")
Rtr, Rte = ratios(tr, S), ratios(te, S)
for col in ("s120", "s122"): print(f"  {col}: raw KS {ks(tr[col], te[col]):.3f}   per-usage KS {ks(Rtr[col + '_perU'], Rte[col + '_perU']):.3f}")

same-bin KS of the five un-shifted histograms (max over bins): {'h23': 0.029, 'h37': 0.03, 'h80': 0.026, 'h129': 0.027, 'h154': 0.03}


KS of the shape features: median 0.010  max 0.067
usage (operating-time total): train median 2.72e+06  test 2.83e+06  KS 0.029
  s120: raw KS 0.740   per-usage KS 0.680
  s122: raw KS 0.312   per-usage KS 0.556


### 3. Do they help? 3 × 5 repeated CV on total cost, paired vs A
A = v5 (base features in both models). B = classifier + shape. C = classifier + shape + usage ratios. E = cost model + shape (classifier unchanged).

In [7]:
SH, RA = shape(dev), ratios(dev, S)
X = {"base": dev[S], "shape": pd.concat([dev[S], SH], axis=1), "ratio": pd.concat([dev[S], SH, RA], axis=1)}
VAR = {"A": ("base", "base"), "B": ("shape", "base"), "C": ("ratio", "base"), "E": ("base", "shape")}
cost = {v: [] for v in VAR}; r2 = {"base": [], "shape": []}; ll = {k: [] for k in X}; auc = {k: [] for k in X}
for r, fs in enumerate((SEED, 1, 2)):
    Fr = list(StratifiedKFold(5, shuffle=True, random_state=fs).split(np.zeros(len(y)), y))
    oo = {k: np.zeros(len(dev)) for k in X}
    for a, b in Fr:
        for k in oo: oo[k][b] = xgb.XGBClassifier(**PARAMS).fit(X[k].iloc[a], y[a]).predict_proba(X[k].iloc[b])[:, 1]
    for k in oo: ll[k].append(log_loss(y, oo[k])); auc[k].append(roc_auc_score(y, oo[k]))
    for a, b in Fr:
        fa, fb = a[y[a] == 1], b[y[b] == 1]; cm = {}
        for k in ("base", "shape"):
            mb, resid = cost_model(CM, X[k].iloc[fa], np.log(c[fa]), X[k].iloc[b]); cm[k] = (mb, resid)
            r2[k].append(r2_score(np.log(c[fb]), mb[y[b] == 1]))
        for v, (fk, ck) in VAR.items():
            fl, bu = decide_pv(make_cal("platt").fit(oo[fk][a], y[a]).predict(oo[fk][b]), *cm[ck]); cost[v].append(per_vehicle(c[b], fl, bu))
A0 = np.array(cost["A"])
print(f"cost-model R2 (log): base {np.mean(r2['base']):.3f}   with shape {np.mean(r2['shape']):.3f}")
for k in X: print(f"classifier on {k:6s}: log-loss {np.mean(ll[k]):.5f}   AUC {np.mean(auc[k]):.4f}")
print("\nvariant  CV cost (15 folds)   paired delta ± se       folds better")
for v in VAR:
    x = np.array(cost[v]); d = x - A0
    print(f"{v}        {x.mean():.4f} ± {x.std():.3f}     {d.mean():+.4f} ± {d.std(ddof=1) / np.sqrt(len(d)):.4f}     {int((d < 0).sum())}/15")

cost-model R2 (log): base 0.658   with shape 0.787
classifier on base  : log-loss 0.01652   AUC 0.9846
classifier on shape : log-loss 0.01683   AUC 0.9862
classifier on ratio : log-loss 0.01692   AUC 0.9853

variant  CV cost (15 folds)   paired delta ± se       folds better
A        0.8870 ± 0.185     +0.0000 ± 0.0000     0/15
B        0.9225 ± 0.219     +0.0355 ± 0.0193     2/15
C        0.9298 ± 0.250     +0.0428 ± 0.0252     4/15
E        0.8664 ± 0.161     -0.0206 ± 0.0283     10/15


The total-cost gain of E is within noise, so check the cost model on its **own** metrics, which are much less noisy: per-fold R² and the budget's own loss (pinball loss at the 0.909 quantile, i.e. the budget's loss for a vehicle that certainly fails).

In [8]:
r2d, pin = {k: [] for k in X if k != "ratio"}, {k: [] for k in X if k != "ratio"}; dR = []
for fs in (SEED, 1, 2):
    for a, b in StratifiedKFold(5, shuffle=True, random_state=fs).split(np.zeros(len(y)), y):
        fa, fb = a[y[a] == 1], b[y[b] == 1]; lc = np.log(c[fa]); out = {}
        for k in ("base", "shape"):
            mb, resid = cost_model(CM, X[k].iloc[fa], lc, X[k].iloc[fb])
            out[k] = (r2_score(np.log(c[fb]), mb), np.exp(mb + np.quantile(resid, Q_STAR)))
            pin[k].append(np.mean(UNDER * np.maximum(c[fb] - out[k][1], 0) + OVER * np.maximum(out[k][1] - c[fb], 0)))
        dR.append(out["shape"][0] - out["base"][0])
dR, dp = np.array(dR), np.array(pin["shape"]) - np.array(pin["base"])
print(f"R2 improves in {int((dR > 0).sum())}/15 folds (mean {dR.mean():+.3f}, worst {dR.min():+.3f})")
print(f"budget loss per failing vehicle: base {np.mean(pin['base']):.2f}  shape {np.mean(pin['shape']):.2f}  (better in {int((dp < 0).sum())}/15 folds)")

R2 improves in 15/15 folds (mean +0.130, worst +0.063)
budget loss per failing vehicle: base 29.97  shape 24.30  (better in 13/15 folds)


### 4. Files (v2 classifier, Platt, cost model with shape features, shift k=4)

In [9]:
oof = fit_oof(dev[S]); p_te = xgb.XGBClassifier(**PARAMS).fit(tr[S], tr.failure).predict_proba(te[S])[:, 1]
q_te = Platt().fit(oof, y).predict(p_te)
f_all = tr[tr.failure == 1]
m_te, resid = cost_model(CM, pd.concat([f_all[S], shape(f_all)], axis=1), np.log(f_all.repair_cost.to_numpy()), pd.concat([te[S], Ste], axis=1))
fl, bu = decide_pv(shift(q_te, 4), m_te, resid); write_submission(te.id, fl, bu, "v8_k4")

wrote outputs/submission_v8_k4.csv: flagged 9.34% of test vehicles


## Results and decisions

**Discovery.** The competition files contain **six 10-bin histograms** (`s023-032, s037-046, s054-063, s080-089, s129-138, s154-163`). Each one's total is a fixed fraction of the reference total for every vehicle (0.296, 0.180, 1.000, 0.749, 0.695, 0.246; spread ±0.001): histograms over a shared **operating time ("usage")**. `s080-s089`, the cost-driving sensors of v5, is one of them. None except `s054-s063` is shifted in test (same-bin KS ≤ 0.03).

**Usage does not explain the depot shift:** it barely differs between depots (median 2.72M vs 2.83M, KS 0.029), and dividing `s120`/`s122` by usage does not remove their drift (KS 0.68 / 0.56).

| Variant (3×5 CV) | CV cost | Paired Δ vs A | Better |
|---|---|---|---|
| A: v5 | 0.887 | n/a | n/a |
| B: classifier + shape | 0.923 | +0.036 ± 0.019 | 2/15 |
| C: classifier + shape + usage ratios | 0.930 | +0.043 ± 0.025 | 4/15 |
| **E: cost model + shape** | **0.866** | **−0.021 ± 0.028** | **10/15** |

- **Cost model with shape features (E): R² 0.66 → 0.79, better in 15/15 folds** (worst gain +0.06); the budget's own loss falls 30.0 → 24.3 (−19%, 13/15). **Kept.**
- **Shape features in the classifier (B, C) hurt:** AUC rises slightly (0.985 → 0.986) but cost gets worse in 13/15 folds. **Rejected**: the third time an AUC/log-loss gain did not turn into a cost gain.

**Caveats:** the structure is inferred from the data (nothing in the course material documents it); no outside data is used. Repeated CV reuses the same 397 failures, so the 15 paired differences are correlated and the standard errors optimistic. Costs are simulated; the large R² jump may reflect how the simulator generates cost from these sensors. The total-cost gain of E alone is within noise; it is kept because its own metrics improve in every fold and the mechanism (repair cost depends on the shape of the s080-s089 histogram) is clear.